# Day 10: Efficient Inference — KV Cache Timing

Welcome to practical 10! Measure latency with and without key/value caching during autoregressive decoding.

This notebook follows the MIT lab style: abstract base classes, PyTorch, and LaTeX in markdown. Wherever you see a `# YOUR CODE HERE` marker, replace the `raise NotImplementedError(...)` below it with your own implementation.

In [ ]:
import math
from abc import ABC, abstractmethod
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

device = torch.device("cpu")
torch.manual_seed(42)
np.random.seed(42)

During decoding, past keys and values can be cached so each new token only computes attention for the latest query. Without caching, cost grows $O(T^2)$ per step; with caching, $O(T)$ per step.

In [ ]:
class TinyDecoder(nn.Module):
    def __init__(self, vocab=64, d_model=64, nhead=4, n_layers=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(512, d_model)
        layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward=128, batch_first=True)
        self.decoder = nn.TransformerDecoder(layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, vocab)

    def forward(self, tgt, memory, tgt_mask=None, past_kv=None):
        B, T = tgt.shape
        pos = torch.arange(T, device=tgt.device)
        x = self.emb(tgt) + self.pos(pos)
        out = self.decoder(x, memory, tgt_mask=tgt_mask)
        return self.head(out)

vocab_size = 64
model = TinyDecoder(vocab=vocab_size).to(device)
memory = torch.randn(1, 8, 64)  # fake encoder output

### Exercise 10.1: Naive autoregressive loop

**Your job**: Re-run full sequence each step.

In [ ]:
import time

@torch.no_grad()
def decode_naive(model, memory, prompt_len=4, gen_len=32):
    seq = torch.randint(0, vocab_size, (1, prompt_len), device=device)
    t0 = time.perf_counter()
    for _ in range(gen_len):
        T = seq.size(1)
        mask = torch.triu(torch.ones(T, T, device=device), diagonal=1).bool()
        logits = model(seq, memory, tgt_mask=mask)
        next_tok = logits[:, -1].argmax(dim=-1, keepdim=True)
        seq = torch.cat([seq, next_tok], dim=1)
    return time.perf_counter() - t0

naive_time = decode_naive(model, memory)
print(f"Naive decode time: {naive_time*1000:.2f} ms")

### Exercise 10.2: Manual KV cache

**Your job**: Store keys/values and append one token at a time.

In [ ]:
class CachedDecoder(nn.Module):
    """Wrapper that caches projected K/V per layer (simplified)."""
    def __init__(self, base: TinyDecoder):
        super().__init__()
        self.base = base

    @torch.no_grad()
    def decode_step(self, token, memory, cache=None):
        # YOUR CODE HERE — single-token forward; cache stores full tgt embeddings so far
        raise NotImplementedError("TODO: complete this exercise")

cached = CachedDecoder(model)

@torch.no_grad()
def decode_cached(model, memory, prompt_len=4, gen_len=32):
    seq = torch.randint(0, vocab_size, (1, prompt_len), device=device)
    cache = None
    t0 = time.perf_counter()
    for _ in range(gen_len):
        logits, cache = model.decode_step(seq[:, -1:], memory, cache)
        next_tok = logits.argmax(dim=-1)
        seq = torch.cat([seq, next_tok], dim=1)
    return time.perf_counter() - t0

cached_time = decode_cached(cached, memory)
print(f"Cached decode time: {cached_time*1000:.2f} ms")

### Exercise 10.3: Benchmark sweep

**Your job**: Compare times across generation lengths.

In [ ]:
lengths = [8, 16, 32, 64]
naive_times, cached_times = [], []
for g in lengths:
    # YOUR CODE HERE
    raise NotImplementedError("TODO: complete this exercise")

plt.plot(lengths, naive_times, marker="o", label="naive")
plt.plot(lengths, cached_times, marker="o", label="cached")
plt.xlabel("tokens generated")
plt.ylabel("seconds")
plt.legend()
plt.title("KV cache timing (simplified)")
plt.show()
print(f"Speedup at 64 tokens: {naive_times[-1]/cached_times[-1]:.2f}x")

### Exercise 10.4: Discuss complexity

**Your job**: Fill in the big-O comparison below.

**Without cache**: each step recomputes attention over all $T$ tokens → $\sum_{T} O(T^2) = O(G^3)$ for $G$ generated tokens.

**With cache**: each step is $O(T)$ → $\sum_T O(T) = O(G^2)$.

Modern LLMs also use FlashAttention and fused kernels; caching is necessary but not sufficient.

## Reflection (Day 10)

Take a few minutes to answer in your own words:

1. What was the most important concept you practiced today (efficient autoregressive inference)?
2. Where did you get stuck, and how did you resolve it?
3. How would you explain today's material to a classmate in two sentences?
4. What would you like to explore further?